# 유방암 데이터셋

---

## 데이터셋의 목적

이 데이터셋의 주 목적은 **유방 종양의 미세침 흡인 세포(FNA) 이미지에서 추출한 특징들을 바탕으로, 해당 종양이 악성(Malignant)인지 양성(Benign)인지 예측**하는 것입니다.


* **`malignant` (악성, 암이 맞음):** 보통 머신러닝에서 레이블 `0`으로 표현됩니다.
* **`benign` (양성, 단순 종양):** 보통 머신러닝에서 레이블 `1`으로 표현됩니다.

> ⚠️ **주의:** 사이킷런 데이터셋에서는 특이하게 **양성(Benign)이 1, 악성(Malignant)이 0**으로 매핑되어 있습니다. 보통 질병 유무를 따질 때 질병이 있는 상태를 `1`로 두는 경우가 많아 헷갈리기 쉬우니 데이터 분석 시 꼭 확인해야 합니다.

---

## 데이터 구조 및 스펙

전체적인 데이터의 크기와 구조는 다음과 같습니다.

* **전체 샘플 수 (행, Rows):** 569개
* 악성(Malignant): 212개
* 양성(Benign): 357개


* **특징 수 (열, Columns/Features):** 30개

---

## 30개의 특징(Features)은 어떻게 만들어졌을까?

종양 세포 핵의 디지털 이미지에서 **10가지 핵심 측정값**을 컴퓨터로 계산해 낸 것입니다.

| 핵심 측정값 (10개) | 설명 |
| --- | --- |
| **`radius`** | 중심에서 외곽선까지의 거리 평균 |
| **`texture`** | 회색조(Gray-scale) 명암 값의 표준편차 |
| **`perimeter`** | 종양의 둘레 길이 |
| **`area`** | 종양의 면적 |
| **`smoothness`** | 반경 길이의 국소적 변화 (부드러운 정도) |
| **`compactness`** | $\frac{\text{perimeter}^2}{\text{area}} - 1.0$ (조밀한 정도) |
| **`concavity`** | 윤곽선에서 오목한 부분의 심한 정도 |
| **`concave points`** | 윤곽선 중 오목한 부분의 개수 |
| **`symmetry`** | 대칭성 |
| **`fractal dimension`** | 프랙탈 차원 (외곽선의 복잡도) |

이 **10가지 측정값** 각각에 대해 통계적인 수치인 **평균(mean), 표준오차(error), 가장 큰 값의 평균(worst)** 3가지를 곱하여 총 30개($10 \times 3$)의 피처가 구성됩니다.

* *예시: `mean radius`, `radius error`, `worst radius` ... 이런 식으로 30개가 존재합니다.*


In [1]:
from sklearn.datasets import load_breast_cancer

# ── 데이터 로드 ──────────────────────────────────────────────────
data = load_breast_cancer()
print("target_names:", data.target_names)
# ['malignant' 'benign']  ← index 0=악성, index 1=정상

# ⚠ 주의: target=0이 malignant(악성), target=1이 benign(정상)!
# 직관과 반대 → y = 1 - data.target 으로 뒤집어서 사용
# y=1: 악성(malignant), y=0: 정상(benign)
y_cancer = 1 - data.target
X_cancer = data.data

target_names: ['malignant' 'benign']


## 2. 데이터 나누기 (훈련용 vs 시험용)

우리가 공부할 때 '연습문제'를 풀고 나서 '실전 시험'을 보는 것처럼, 인공지능도 똑같이 학습을 합니다.
* **훈련 데이터 (Train set):** 연습문제 (80%)
* **테스트 데이터 (Test set):** 실전 시험문제 (20%)

> 💡 **중요! (계층적 분할, stratify)**
> 암 환자와 정상 환자의 비율이 한쪽으로 쏠리지 않도록, 연습문제와 실전 시험문제에 골고루 섞어주는 기술을 `stratify`라고 부릅니다.

> 💡 **중요! (데이터 스케일링, StandardScaler)**
> 종양의 '면적'은 500, 1000처럼 숫자가 엄청 큰데, '부드러움 정도'는 0.05처럼 아주 작습니다. 숫자의 크기가 너무 차이나면 인공지능이 헷갈려합니다. 그래서 모든 숫자의 크기(스케일)를 비슷하게 맞춰주는 작업이 꼭 필요합니다!

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 데이터 쪼개기 (8:2 비율, 악성/양성 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

# 스케일링 (숫자 크기 맞추기)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # 훈련 데이터로 기준을 잡고 변환
X_test_scaled = scaler.transform(X_test)       # 테스트 데이터는 그 기준에 맞춰서 변환만!

print("데이터 나누기 및 스케일링 완료!")
print(f"훈련용 데이터: {X_train_scaled.shape[0]}개")
print(f"시험용 데이터: {X_test_scaled.shape[0]}개")

데이터 나누기 및 스케일링 완료!
훈련용 데이터: 455개
시험용 데이터: 114개


## 3. 인공지능 학습시키기 (로지스틱 회귀)

로지스틱 회귀는 이름은 '회귀(Regression)'지만, 실제로는 "이게 암일까? 아닐까?"를 확률로 계산해서 **분류**해주는 훌륭한 알고리즘입니다.

In [3]:
from sklearn.linear_model import LogisticRegression

# 로지스틱 회귀 모델 만들기
lr = LogisticRegression(random_state=42)

# 인공지능 학습시키기 (훈련용 데이터로 공부!)
lr.fit(X_train_scaled, y_train)

# 공부를 얼마나 잘했는지 점수 확인
train_score = lr.score(X_train_scaled, y_train)
test_score = lr.score(X_test_scaled, y_test)

print(f"연습문제(Train) 점수: {train_score * 100:.2f} 점")
print(f"실전시험(Test) 점수: {test_score * 100:.2f} 점")

연습문제(Train) 점수: 98.68 점
실전시험(Test) 점수: 96.49 점


## 4. 진짜 중요한 건 점수가 아니라 '놓친 암 환자' 찾기!

점수가 97점이라고 무조건 좋아하면 안 됩니다. 
병원에서는 **"실제 암 환자인데, 정상이라고 잘못 판단해서 돌려보내는 것(놓침, False Negative)"**이 가장 치명적인 실수입니다.

이걸 한눈에 보기 위해 **오차 행렬(Confusion Matrix)**이라는 4칸짜리 성적표를 확인해 봅니다.

In [6]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
from IPython.display import display

# 인공지능이 실전 시험을 풀게 합니다
y_pred = lr.predict(X_test_scaled)

# 오차 행렬 만들기
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm, 
    index=['실제 양성(0, 단순종양)', '실제 악성(1, 암)'], 
    columns=['예측 양성(0)', '예측 악성(1)']
)
print("--- 오차 행렬 (기본 커트라인 50% 적용) ---")
display(cm_df)

# 악성을 놓친 수 (FN)
fn_count = cm[1, 0]
print(f"\n🚨 실제 암 환자인데 정상이라고 잘못 판단해서 집에 돌려보낸 환자(FN) 수: {fn_count}명")

--- 오차 행렬 (기본 커트라인 50% 적용) ---


,예측 양성(0),예측 악성(1)
"실제 양성(0, 단순종양)",71,1
"실제 악성(1, 암)",3,39



🚨 실제 암 환자인데 정상이라고 잘못 판단해서 집에 돌려보낸 환자(FN) 수: 3명


## 5. 진짜 암 환자를 찾을 확률을 높이는 전략 (커트라인 낮추기)

인공지능은 기본적으로 **"내가 50% 이상 확신할 때만 암이라고 예측할게"**라는 기준(임계값)을 갖고 있습니다.

하지만 생명이 걸린 일이라면? 
**"야, 30%만 의심스러워도 일단 암이라고 예측하고 정밀 검사 받게 해!"**라고 기준(커트라인)을 팍 낮춰버릴 수 있습니다.

이렇게 하면 정상인 사람을 암이라고 오해하는 경우(거짓 알람, FP)는 늘어나겠지만, **진짜 암 환자를 놓치는 일(FN)은 확 줄일 수 있습니다.** 이것이 바로 **재현율(Recall)을 끌어올리는 전략**입니다.

In [7]:
import numpy as np

# 각 환자가 '암일 확률'을 모두 구합니다
y_prob = lr.predict_proba(X_test_scaled)[:, 1] # 1(악성)일 확률만 쏙 빼옵니다

# 커트라인을 50%에서 30%로 낮춥니다 (확률이 0.3 이상이면 무조건 암(1)으로 판정!)
y_pred_custom = (y_prob >= 0.3).astype(int)

# 새로운 오차 행렬 만들기
cm_custom = confusion_matrix(y_test, y_pred_custom)
cm_custom_df = pd.DataFrame(
    cm_custom, 
    index=['실제 양성(0, 단순종양)', '실제 악성(1, 암)'], 
    columns=['예측 양성(0)', '예측 악성(1)']
)

print("--- 오차 행렬 (안전제일주의! 커트라인 30% 적용) ---")
display(cm_custom_df)

# 새롭게 악성을 놓친 수 (FN)
fn_custom_count = cm_custom[1, 0]
print(f"\n커트라인을 낮췄더니 놓친 암 환자(FN) 수가 {fn_custom_count}명으로 감소.")

print("\n--- 변경된 커트라인의 분류 보고서 ---")
print(classification_report(y_test, y_pred_custom, target_names=['양성(0)', '악성(1)']))

--- 오차 행렬 (안전제일주의! 커트라인 30% 적용) ---


,예측 양성(0),예측 악성(1)
"실제 양성(0, 단순종양)",71,1
"실제 악성(1, 암)",1,41



커트라인을 낮췄더니 놓친 암 환자(FN) 수가 1명으로 감소.

--- 변경된 커트라인의 분류 보고서 ---
              precision    recall  f1-score   support

       양성(0)       0.99      0.99      0.99        72
       악성(1)       0.98      0.98      0.98        42

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

